In [1]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
batch_size = 16
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [3]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


In [4]:
for layer in base_model.layers:
    layer.trainable = False


In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)  # covid, normal, pneumonia

model = Model(inputs=base_model.input, outputs=output)


In [6]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [7]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 187s 603ms/step - accuracy: 0.6935 - loss: 0.7572 - val_accuracy: 0.8448 - val_loss: 0.4722
Epoch 2/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 240s 792ms/step - accuracy: 0.8265 - loss: 0.4709 - val_accuracy: 0.8739 - val_loss: 0.3853
Epoch 3/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 298s 984ms/step - accuracy: 0.8527 - loss: 0.4016 - val_accuracy: 0.8846 - val_loss: 0.3635
Epoch 4/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 267s 881ms/step - accuracy: 0.8759 - loss: 0.3556 - val_accuracy: 0.8875 - val_loss: 0.3357
Epoch 5/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 281s 928ms/step - accuracy: 0.8776 - loss: 0.3459 - val_accuracy: 0.8923 - val_loss: 0.3210
Epoch 6/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 266s 877ms/step - accuracy: 0.8920 - loss: 0.3166 - val_accuracy: 0.8962 - val_loss: 0.3228
Epoch 7/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 257s 849ms/step - accuracy: 0.8960 - loss: 0.3154 - val_accuracy: 0.8972 - val_loss: 0.3108
Epoch 8/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 6693s 22s/step - accuracy: 0.8968 - 

In [8]:
for layer in base_model.layers[-40:]:
    layer.trainable = True


In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [10]:
history_fine = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)


Epoch 1/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 188s 601ms/step - accuracy: 0.7510 - loss: 0.8510 - val_accuracy: 0.9020 - val_loss: 0.2950
Epoch 2/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 210s 693ms/step - accuracy: 0.8900 - loss: 0.3212 - val_accuracy: 0.9030 - val_loss: 0.2925
Epoch 3/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 222s 733ms/step - accuracy: 0.8867 - loss: 0.3144 - val_accuracy: 0.9069 - val_loss: 0.2838
Epoch 4/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 238s 784ms/step - accuracy: 0.8968 - loss: 0.3038 - val_accuracy: 0.9079 - val_loss: 0.2794
Epoch 5/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 251s 829ms/step - accuracy: 0.9011 - loss: 0.2781 - val_accuracy: 0.9098 - val_loss: 0.2735
Epoch 6/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 248s 818ms/step - accuracy: 0.9053 - loss: 0.2698 - val_accuracy: 0.9127 - val_loss: 0.2650
Epoch 7/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 247s 816ms/step - accuracy: 0.9034 - loss: 0.2616 - val_accuracy: 0.9127 - val_loss: 0.2614
Epoch 8/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 252s 830ms/step - accuracy: 0.9179 -

In [11]:
test_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
model.evaluate(test_gen)


Found 1036 images belonging to 3 classes.
65/65 ━━━━━━━━━━━━━━━━━━━━ 43s 667ms/step - accuracy: 0.9344 - loss: 0.1843


[0.18427477777004242, 0.9343629479408264]